<a href="https://colab.research.google.com/github/dongyah/EA2_SCY1101_Calidad_Vino/blob/main/05_final_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 5 - Análisis Final: Aprendizaje No Supervisado

En las fases anteriores logramos predecir la calidad del vino con Aprendizaje Supervisado. Sin embargo, para cumplir con el indicador IEE 2.1.2 de la rúbrica, en esta fase aplicaré **Aprendizaje No Supervisado**.

Mi objetivo aquí es ocultarle la etiqueta de calidad (`quality`) al modelo y usar algoritmos de Clustering (K-Means) y Reducción de Dimensionalidad (PCA) para descubrir si existen 'familias' o patrones químicos ocultos en los vinos de forma natural.

In [ ]:
!git clone https://github.com/dongyah/EA2_SCY1101_Calidad_Vino.git
%cd EA2_SCY1101_Calidad_Vino

### 1. Preparar los datos (Sin trampas)

**Justificación:** En el aprendizaje no supervisado no hay 'respuestas correctas'. Por lo tanto, elimino la columna `quality`. Además, como algoritmos como K-Means funcionan calculando distancias físicas entre los puntos, es estrictamente obligatorio escalar los datos con `StandardScaler` para que variables con números grandes (como el azufre) no dominen el agrupamiento.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

df = pd.read_csv('data/processed/winequality_clean.csv')

# Le quito la etiqueta de respuesta correcta al modelo
X_sin_etiquetas = df.drop('quality', axis=1)

# Escalo los datos a mano para que tengan media 0 y varianza 1
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X_sin_etiquetas)
print("Datos listos y escalados.")

### 2. Buscar el número ideal de grupos: El Método del Codo

**Justificación:** Como no sé cuántas 'familias' de vino existen realmente, probaré agruparlos desde 1 hasta 10 clusters midiendo su **Inercia** (la tensión o distancia de los puntos hacia el centro de su grupo). Buscaré el punto en el gráfico donde la caída deje de ser brusca (el 'Codo').

In [ ]:
inercia = []

# Pruebo de 1 a 10 grupos
for k in range(1, 11):
    modelo_kmeans = KMeans(n_clusters=k, random_state=42)
    modelo_kmeans.fit(X_escalado)
    inercia.append(modelo_kmeans.inertia_)

# Dibujo el gráfico del codo
plt.figure(figsize=(7,4))
plt.plot(range(1, 11), inercia, marker='o', color='red')
plt.title('Método del Codo para buscar el K ideal')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inercia')
plt.grid(True)
plt.show()

### 3. Aplicar K-Means y evaluar con Silhouette Score

**Justificación:** Observando el gráfico anterior, el 'codo' se forma en K=4. Por lo tanto, configuraré el algoritmo `K-Means` para encontrar 4 perfiles químicos de vino.

Para saber si estos 4 grupos son realmente buenos, no puedo usar *Accuracy*. Usaré el **Silhouette Score**, que mide qué tan bien separado está cada grupo de los demás (cercano a 1 es perfecto, cercano a 0 indica que se solapan).

In [ ]:
# Aplico K-Means con 4 grupos
kmeans_final = KMeans(n_clusters=4, random_state=42)
etiquetas_clusters = kmeans_final.fit_predict(X_escalado)

# Le pido la nota de silueta
nota_silueta = silhouette_score(X_escalado, etiquetas_clusters)
print(f"El Silhouette Score para K=4 es: {nota_silueta:.4f}")

### 4. Reducción de Dimensionalidad con PCA y Mapa Final

**Justificación Técnica:** Nuestro dataset cuenta con 11 variables químicas distintas. Graficar el agrupamiento que hizo K-Means requeriría un gráfico de 11 dimensiones, lo cual es humanamente imposible de visualizar.

Para resolver esto, aplicamos **PCA (Análisis de Componentes Principales)**. Como hemos visto en el curso, PCA actúa como un reductor de dimensionalidad: comprime las 11 variables en solo 2 componentes principales (PCA 1 y PCA 2), conservando la mayor cantidad de la varianza original o "esencia matemática" de los datos. Esto nos permite proyectar los clústeres en un plano 2D para que el cerebro humano pueda interpretar visualmente los patrones.

**¿Cómo interpretar el siguiente gráfico?**
* Cada punto representa una botella de vino.
* Los colores representan a las 4 "familias" o perfiles químicos distintos que el algoritmo K-Means descubrió por sí solo.
* Si los puntos de un mismo color se agrupan de forma cercana formando "vecindarios", significa que esas botellas comparten un ADN químico muy similar entre sí, confirmando que nuestro agrupamiento fue exitoso.

In [ ]:
# Comprimo las 11 columnas a solo 2
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_escalado)

# Grafico el mapa final
plt.figure(figsize=(9,6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=etiquetas_clusters, cmap='viridis', alpha=0.7)

plt.title('Cartografía de Vinos: 4 Perfiles Químicos detectados con PCA', fontsize=14)
plt.xlabel('PCA 1 (Resumen Químico Principal)')
plt.ylabel('PCA 2 (Resumen Químico Secundario)')
plt.colorbar(scatter, label='Grupo (Cluster)')
plt.grid(True, alpha=0.3)
plt.show()

### 5. Extra: Identificando el Clúster "Premium" (El VIP de los Vinos)

**Justificación de Negocio:** En clases vimos cómo el Aprendizaje No Supervisado sirve para encontrar perfiles estratégicos, como los "Clientes VIP" de un Mall. Aplicaremos la misma lógica aquí.

Aunque nuestro algoritmo K-Means agrupó los vinos **sin saber su nota de calidad** (estudió a ciegas), ahora cruzaremos los 4 grupos que inventó con la nota real de calidad y el nivel de alcohol para descubrir si logró aislar naturalmente la "fórmula" de los vinos Premium.

In [ ]:
# Le pegamos las etiquetas de los 4 grupos creados al dataset original
df['Grupo_KMeans'] = etiquetas_clusters

# Agrupamos por este nuevo grupo y calculamos el promedio de calidad, alcohol y acidez
perfiles = df.groupby('Grupo_KMeans')[['quality', 'alcohol', 'volatile acidity']].mean().round(2)

# Ordenamos para que el grupo con mejor nota quede arriba
perfiles = perfiles.sort_values(by='quality', ascending=False)

print("--- PERFILES QUÍMICOS DE LAS FAMILIAS DE VINO ---")
print(perfiles)
print("\n¡Éxito! El algoritmo logró aislar a la familia de Vinos Premium basándose solo en su química.")

### Conclusión Final del Proyecto

A lo largo de este proyecto logré demostrar que la calidad del vino tinto no es un proceso al azar.
1. Con el **Aprendizaje Supervisado**, comprobamos que los modelos de Machine Learning (como el Random Forest) pueden aprender a predecir la calidad de un vino con alta exactitud tras una correcta limpieza y ajuste de hiperparámetros.
2. Con el **Aprendizaje No Supervisado**, confirmamos que, incluso sin saber la nota humana, la química del vino por sí sola agrupa a las botellas en 4 familias o perfiles estadísticos bien diferenciados (mostrados en el mapa PCA), permitiéndonos aislar el perfil "Premium".

Dejo el proyecto sincronizado para su evaluación final.

In [ ]:
!git add .
!git commit -m "Fase 5: Proyecto finalizado. PCA, K-Means y descubrimiento de Vinos Premium integrados"
!git push origin main